In [1]:
import torch

In [3]:
weight_fp32 = torch.randn(4,4)

In [4]:
# after creating the fake matrix, i need to print it

In [5]:
print(weight_fp32)

tensor([[-0.2085,  2.2875, -1.4327,  0.3598],
        [ 1.0515,  0.8319, -0.2556, -1.5947],
        [ 0.3340,  0.9820,  0.8277, -0.2207],
        [-0.5041, -1.4667,  0.3189, -0.1544]])


In [7]:
# so now, we need to know how much meemory it is using everything
#  is float so fp32 means 4 bytes

In [8]:
print("memory used : ", weight_fp32.element_size() * weight_fp32.nelement(), "bytes")


memory used :  64 bytes


- so there are 16 elelements each of size 4 bytes
- so 16x 4 = 64

In [9]:
# now i need to convert it to int 8
# to do it i need a scale
# i need to pickup the absolute maximum value
sacle = weight_fp32.abs().max() / 127.0


tensor(
  
        [[-0.2085,  2.2875, -1.4327,  0.3598],

        [ 1.0515,  0.8319, -0.2556, -1.5947],

        [ 0.3340,  0.9820,  0.8277, -0.2207],

        [-0.5041, -1.4667,  0.3189, -0.1544]]
        
        
      )
      

- in the above matrix the greatest value of all is 2.2875
- since we have used randn by the time we run it next time we would have got an other matrix
- so we need to do this in a single run
scale = 2.2875 / 127.0 = 0.01801181102


In [10]:
print("\nScale factor:", scale.item())


Scale factor: 0.01744004525244236


so it was almost so near to what we have calculated

lets now convert it to int 8 which is a number between -127 to 127

In [11]:
weight_int8 = (weight_fp32 / scale ).round().clamp(-128,127).to(torch.int8)

- we have rounded it up to the nearest integer so and also clamped it to avoid overflow .
- It should be in the range of -127 to 127 only


In [12]:
print("\nQuantized INT8 weight:")
print(weight_int8)
print("Memory used:", weight_int8.element_size() * weight_int8.nelement(), "bytes")
# element_size() = 1 byte (INT8)


Quantized INT8 weight:
tensor([[-12, 127, -82,  21],
        [ 60,  48, -15, -91],
        [ 19,  56,  47, -13],
        [-29, -84,  18,  -9]], dtype=torch.int8)
Memory used: 16 bytes


In [13]:
weight_dequantized = weight_int8.to(torch.float32) * scale


In [14]:

print("\nDequantized weight (approximately original):")
print(weight_dequantized)



Dequantized weight (approximately original):
tensor([[-0.2093,  2.2149, -1.4301,  0.3662],
        [ 1.0464,  0.8371, -0.2616, -1.5870],
        [ 0.3314,  0.9766,  0.8197, -0.2267],
        [-0.5058, -1.4650,  0.3139, -0.1570]])


In [15]:
error = (weight_fp32 - weight_dequantized).abs().mean()
print("\nAverage quantization error:", error.item())
# This will be very small — like 0.008 or so


Average quantization error: 0.008714866824448109


- so the error is so minimal that the human wouldnt see any difference in response

- Quantization compresses neural network weights and activations from high-precision formats (like FP32) to lower-precision formats (like INT8) to reduce memory usage and data movement.
- This speeds up inference because more values can fit in cache/SRAM and be transferred from HBM per second.